In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyDs


In [3]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

initialise our model here

In [4]:
MODEL = 'gpt-4.1-mini'

In [5]:
# Removed pick_agent function - manager_agent handles routing internally

In [ ]:
from src.agents.manager_agent import manager_agent
import re

def build_context(message, history):
    """Build conversation context from message and history for the manager agent."""
    if not history:
        return message
    # Format history as a conversation string
    parts = []
    for h in history:
        # Handle both dict format (Gradio) and string format
        if isinstance(h, dict):
            role = h.get('role', 'user')
            content = h.get('content', '')
            if isinstance(content, list):
                # Handle Gradio's list format [{'text': '...', 'type': 'text'}]
                text_parts = [item.get('text', '') for item in content if isinstance(item, dict)]
                content = ' '.join(text_parts)
            parts.append(f"{role}: {content}")
        else:
            parts.append(str(h))
    parts.append(f"user: {message}")
    return "\n".join(parts)

def clean_response(response):
    """Clean up verbose formatting from manager agent responses while preserving content like lists."""
    if not isinstance(response, str):
        return response
    
    original_response = response
    
    # Check if response has the verbose format pattern
    if "Here is the final answer from your managed agent" in response:
        # Extract everything after the header, preserving all content
        # Remove just the header, keep everything else
        cleaned = re.sub(
            r"Here is the final answer from your managed agent '[^']+':\s*", 
            "", 
            response, 
            flags=re.IGNORECASE
        )
        
        # Remove section headers but preserve their content
        cleaned = re.sub(r'### \d+\. Task outcome \([^)]+\):\s*', '', cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r'### \d+\. Additional context[^\n]*:\s*', '', cleaned, flags=re.IGNORECASE)
        
        # If we have multiple sections, prefer the "short version" but include all meaningful content
        if "### 1. Task outcome (short version)" in original_response:
            # Extract short version content
            short_match = re.search(
                r'### 1\. Task outcome \(short version\):\s*(.+?)(?=\n\n### \d+\.|$)', 
                original_response, 
                re.DOTALL | re.IGNORECASE
            )
            if short_match:
                short_content = short_match.group(1).strip()
                # If short content looks complete (has lists, multiple lines, etc.), use it
                if '\n' in short_content or '[' in short_content or '{' in short_content:
                    return short_content
                # Otherwise, check if there's more in other sections
                detailed_match = re.search(
                    r'### 2\. Task outcome \(extremely detailed version\):\s*(.+?)(?=\n\n### \d+\.|$)', 
                    original_response, 
                    re.DOTALL | re.IGNORECASE
                )
                if detailed_match:
                    detailed_content = detailed_match.group(1).strip()
                    # Combine if short is just a summary
                    if len(short_content) < 100 and len(detailed_content) > len(short_content):
                        return detailed_content
                return short_content
        
        return cleaned.strip()
    
    # Remove section headers if present, but preserve the content
    cleaned = re.sub(r'### \d+\. Task outcome \([^)]+\):\s*', '', response, flags=re.IGNORECASE)
    cleaned = re.sub(r'### \d+\. Additional context[^\n]*:\s*', '', cleaned, flags=re.IGNORECASE)
    
    # Detect if response is just a summary without actual content
    summary_indicators = [
        r'A (comprehensive |full |complete )?list.*has been (compiled|retrieved|gathered)',
        r'has been (successfully )?(compiled|retrieved|gathered|created)',
    ]
    
    is_likely_summary = any(re.search(pattern, cleaned, re.IGNORECASE) for pattern in summary_indicators)
    
    # If it's a short summary statement, the actual content might be missing
    # This suggests the manager agent summarized instead of passing through
    if is_likely_summary and len(cleaned) < 150:
        # Return as-is but log that this might be a summary issue
        print(f"--- WARNING: Response appears to be a summary without actual content ---")
        return cleaned.strip()
    
    return cleaned.strip()

def unified_chat(message, history):
    """Unified chat function that uses the manager agent for routing and delegation."""
    task = build_context(message, history)
    print(f"--- SYSTEM: Request handled by manager agent")
    response = manager_agent.run(task)
    # Clean up verbose formatting if present
    return clean_response(response)

In [ ]:
import gradio as gr

view = gr.ChatInterface(
    fn=unified_chat,
    title="SimpliAsk HR Agent",
    description="Check your terminal to see the Agent Traces (routing + tools) in real-time.",
    examples=[
        "I would like to apply for 7 days of annual leave.",
        "I would like to request for HDMI Cable.",
        "Help me submit a medical claim from my receipt.",
        "List workflows you can assist me with.",
    ],
)

view.launch(share=True)

c:\projects\simpliAsk\simpliAsk\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6cba6307a327fdebe2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


--- SYSTEM: Request handled by manager agent


╭───────────────────────────────────────────── New run - hr_manager ──────────────────────────────────────────────╮
│                                                                                                                 │
│ I would like to apply for 7 days of annual leave.                                                               │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

Observations: ### 1. Task outcome (short version):
A full list of available devices for employees to request has been retrieved, including device IDs, names, and 
costs.

### 2. Task outcome (extremely detailed version):
The available devices that employees can request are:
1. Device ID: 1, Name: 2M HDMI Cable, Cost: $6.50
2. Device ID: 2, Name: Wireless Mouse, Cost: $15.00
3. Device ID: 3, Name: Mechanical Keyboard, Cost: $45.00
4. Device ID: 4, Name: 27-inch Monitor, Cost: $230.00
5. Device ID: 5, Name: USB-C Hub, Cost: $25.50
6. Device ID: 6, Name: External Hard Drive 1TB, Cost: $65.00
7. Device ID: 7, Name: Laptop Stand, Cost: $30.00
8. Device ID: 8, Name: Webcam 1080p, Cost: $40.00

The list provides a comprehensive overview of device options, including accessories such as cables, input devices, 
storage, monitors, and webcam, relevant for various employee needs.

### 3. Additional context (if relevant):
The costs likely reflect approximate prices for budgeting and approval purposes; employees can select devices based
on their specific job requirements and preferences.

Final answer: ### 1. Task outcome (short version):
A full list of available devices for employees to request has been retrieved, including device IDs, names, and 
costs.

### 2. Task outcome (extremely detailed version):
The available devices that employees can request are:
1. Device ID: 1, Name: 2M HDMI Cable, Cost: $6.50
2. Device ID: 2, Name: Wireless Mouse, Cost: $15.00
3. Device ID: 3, Name: Mechanical Keyboard, Cost: $45.00
4. Device ID: 4, Name: 27-inch Monitor, Cost: $230.00
5. Device ID: 5, Name: USB-C Hub, Cost: $25.50
6. Device ID: 6, Name: External Hard Drive 1TB, Cost: $65.00
7. Device ID: 7, Name: Laptop Stand, Cost: $30.00
8. Device ID: 8, Name: Webcam 1080p, Cost: $40.00

The list provides a comprehensive overview of device options, including accessories such as cables, input devices, 
storage, monitors, and webcam, relevant for various employee needs.

### 3. Additional context (if relevant):
The costs likely reflect approximate prices for budgeting and approval purposes; employees can select devices based
on their specific job requirements and preferences.

[Step 2: Duration 4.84 seconds| Input tokens: 5,497 | Output tokens: 331]

Final answer: Here is the final answer from your managed agent 'device_specialist':
### 1. Task outcome (short version):
A full list of available devices for employees to request has been retrieved, including device IDs, names, and 
costs.

### 2. Task outcome (extremely detailed version):
The available devices that employees can request are:
1. Device ID: 1, Name: 2M HDMI Cable, Cost: $6.50
2. Device ID: 2, Name: Wireless Mouse, Cost: $15.00
3. Device ID: 3, Name: Mechanical Keyboard, Cost: $45.00
4. Device ID: 4, Name: 27-inch Monitor, Cost: $230.00
5. Device ID: 5, Name: USB-C Hub, Cost: $25.50
6. Device ID: 6, Name: External Hard Drive 1TB, Cost: $65.00
7. Device ID: 7, Name: Laptop Stand, Cost: $30.00
8. Device ID: 8, Name: Webcam 1080p, Cost: $40.00

The list provides a comprehensive overview of device options, including accessories such as cables, input devices, 
storage, monitors, and webcam, relevant for various employee needs.

### 3. Additional context (if relevant):
The costs likely reflect approximate prices for budgeting and approval purposes; employees can select devices based
on their specific job requirements and preferences.

[Step 1: Duration 7.87 seconds| Input tokens: 3,557 | Output tokens: 131]